# 1. 上下文增强

在基础 RAG 中，检索器返回的片段往往「看起来相关，但读起来不够用」——命中了关键句，却丢失了前后的支撑信息。这不是检索没找对，而是切块粒度导致上下文被截断了。

上下文增强要做的事情很简单：**在命中之后，把周围的关键信息补回来**。本节在 **同一份 `train_dataset.json` 子集**（变量 `QA_INDICES`）上，依次对比 Baseline、Sentence Window、Small-to-Big、AutoMerging；既看 **inspect 机制**，也看 **LLM 裁判** 与 **相对 Baseline 是否改善**（见文末对比表）。

> 若修改切块参数或 `QA_INDICES`，请删除对应 `./chroma_db/` 子目录后重跑以重建索引。


## 环境准备

本节使用智谱 AI 的 `GLM-4-Flash` 做生成模型，使用本地 `BAAI/bge-small-zh-v1.5` 做 embedding。运行前请确保：

1. 安装依赖：`pip install langchain langchain-community langchain-chroma zhipuai python-dotenv pymupdf pandas modelscope sentence-transformers transformers torch`
2. 在项目根目录的 `.env` 文件中配置 `ZHIPUAI_API_KEY`
3. 首次运行会自动从 ModelScope 下载本地 embedding 模型到当前目录下的 `./models/`

> **数据说明**：本教程使用与「3. 索引阶段」相同的数据集（南瓜书《机器学习公式详解》），保持教程连贯性。

## 统一实验设置（一次定义，后面复用）

- 数据：`../3. 索引阶段/data/pumpkin_book.pdf`（南瓜书《机器学习公式详解》）
- 问答：`../3. 索引阶段/data/train_dataset.json`
- **共用测试子集**：`QA_INDICES`（Baseline 与三种增强同一批题，不另建题表）
- 生成模型：`glm-4-flash-250414`
- 向量模型：本地 `BAAI/bge-small-zh-v1.5`
- 对比：`baseline_df` 与 `sentence_window_df` / `small_to_big_df` / `auto_merging_df`
- 评估：LLM 裁判（偏严）；表格为辅助，并结合文末 **同题对比** 阅读


In [ ]:
import os
import re
import json
import time
import warnings
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain_community.chat_models import ChatZhipuAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_chroma import Chroma
from modelscope import snapshot_download

warnings.filterwarnings("ignore")
load_dotenv()

api_key = os.environ.get("ZHIPUAI_API_KEY")
llm = ChatZhipuAI(model="glm-4-flash-250414", temperature=0.0, api_key=api_key)

EMBED_MODEL_ID = "BAAI/bge-small-zh-v1.5"
EMBED_MODEL_PATH = f"./models/{EMBED_MODEL_ID}"

if not os.path.exists(EMBED_MODEL_PATH):
    EMBED_CACHE_DIR = Path("./models")
    EMBED_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    EMBED_MODEL_PATH = snapshot_download(EMBED_MODEL_ID, cache_dir=str(EMBED_CACHE_DIR))

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL_PATH,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

PDF_PATH = "../3. 索引阶段/data/pumpkin_book.pdf"
QA_PATH = "../3. 索引阶段/data/train_dataset.json"

def clean_text(text: str):
    """文本清理函数"""
    text = re.sub(r'→_→\n欢迎去各大电商平台选购纸质版南瓜书《机器学习公式详解》\n←_←', '', text)
    text = re.sub(r'→_→\n配套视频教程：https://www.bilibili.com/video/BV1Mh411e7VU\n←_←', '', text)
    text = re.sub(r'\s+', '', text)
    text = re.sub(r'\n+', '', text)
    return text

def split_sentences_for_window(text: str, max_piece: int = 320):
    """按句切分；过长片段（多为目录等）再切段，避免窗口扩进整段目录。"""
    raw = [s.strip() for s in re.split(r"(?<=[。！？!?])", text) if s.strip()]
    out = []
    for s in raw:
        if len(s) <= max_piece:
            out.append(s)
        else:
            for i in range(0, len(s), max_piece):
                out.append(s[i : i + max_piece])
    return out

def llm_call(prompt):
    """带重试和节流的 LLM 调用"""
    last_error = None
    for attempt in range(10):
        try:
            result = llm.invoke(prompt).content
            time.sleep(20)
            return result
        except Exception as e:
            last_error = e
            if '429' in str(e) and attempt < 9:
                wait = min(180, 20 * (attempt + 1))
                print(f"  [速率限制，等待 {wait}s...]")
                time.sleep(wait)
            else:
                raise
    raise last_error

def build_chroma_from_docs(chunks, emb, persist_directory=None):
    if persist_directory and os.path.exists(persist_directory) and os.listdir(persist_directory):
        print(f"  -> 加载已有索引: {persist_directory}")
        return Chroma(persist_directory=persist_directory, embedding_function=emb)
    if persist_directory:
        print(f"  -> 创建新索引: {persist_directory}")
    return Chroma.from_documents(chunks, embedding=emb, persist_directory=persist_directory)

def build_chroma_from_texts(texts, emb, persist_directory=None):
    if persist_directory and os.path.exists(persist_directory) and os.listdir(persist_directory):
        print(f"  -> 加载已有索引: {persist_directory}")
        return Chroma(persist_directory=persist_directory, embedding_function=emb)
    if persist_directory:
        print(f"  -> 创建新索引: {persist_directory}")
    return Chroma.from_texts(texts, embedding=emb, persist_directory=persist_directory)

def load_chunks(chunk_size=256, chunk_overlap=20):
    docs = PyMuPDFLoader(PDF_PATH).load()
    for doc in docs:
        doc.page_content = clean_text(doc.page_content)
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_documents(docs)

def build_retriever(chunk_size=256, chunk_overlap=20, k=4):
    persist_dir = f"./chroma_db/baseline_{chunk_size}_{chunk_overlap}"
    if os.path.exists(persist_dir) and os.listdir(persist_dir):
        vs = build_chroma_from_docs(None, embeddings, persist_directory=persist_dir)
    else:
        chunks = load_chunks(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        vs = build_chroma_from_docs(chunks, embeddings, persist_directory=persist_dir)
    return vs.as_retriever(search_kwargs={"k": k})

with open(QA_PATH, 'r', encoding='utf-8') as f:
    qa_pairs = json.load(f)

# 共用测试子集（train 下标，0-based）
QA_INDICES = [0, 5, 14, 26, 27]

# 各方法「典型胜场」题下标：跑完后在对比单元格中自动核对（不抛错，仅提示）
WIN_TARGETS = {"Sentence Window": 5, "Small-to-Big": 0, "AutoMerging": 26}

def build_ordered_qa_dict(pairs, indices):
    out = {}
    for i in indices:
        item = pairs[i]
        q, a = item["query"], item["answer"]
        if q and str(q).strip():
            out[q] = a
    return out

qna_dict = build_ordered_qa_dict(qa_pairs, QA_INDICES)
question_to_train_idx = {qa_pairs[i]["query"]: i for i in QA_INDICES if qa_pairs[i].get("query")}

print(f"✅ 加载 {len(qa_pairs)} 条问答；本实验共用 {len(qna_dict)} 题，train 下标 {QA_INDICES}")

def simple_eval(llm_answer: str, expected_answer: str, question: str = "") -> str:
    """LLM 裁判（偏严）；少要点即 ❌。"""
    prompt = (
        "请作为一名严谨的判卷人，评估模型给出的答案是否回答了用户的问题，并且与参考答案的核心意思完全一致。\n"
        "如果模型答案遗漏了参考答案中的任何一点（例如少了一种方法、少了一个角度），请判定为\"❌\"。\n\n"
        f"用户问题：{question}\n"
        f"参考答案：{expected_answer}\n"
        f"模型答案：{llm_answer}\n\n"
        "请仅输出以下两种结果之一，不要输出任何其他解释：\n"
        "- ✅\n"
        "- ❌"
    )
    try:
        result = llm_call(prompt).strip()
        return "✅" if "✅" in result else "❌"
    except Exception as e:
        print(f"评估失败: {e}")
        return "❌"

print("✅ 环境准备完成")


## Baseline：纯向量检索先跑一遍

先不做任何上下文增强，只用基础向量检索回答 `qna_dict`。

失败观察重点：
- 命中内容是否相关；
- 最终回答是否相关但不完整。

In [ ]:
# Baseline：小块检索，突出「命中但不完整」
baseline_retriever = build_retriever(chunk_size=96, chunk_overlap=12, k=4)

baseline_rows = []
for question, expected in qna_dict.items():
    docs = baseline_retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs)
    prompt = f"""
仅根据上下文回答问题，如果上下文没有包含完整答案，请仅回答上下文中的内容，不要补充你自己的知识。

问题：{question}
上下文：
{context}
"""
    answer = llm_call(prompt)
    baseline_rows.append({
        "train_idx": question_to_train_idx[question],
        "question": question,
        "llm_answer": answer,
        "expected_answer": expected,
        "rag_eval_results": simple_eval(answer, expected, question),
    })

baseline_df = pd.DataFrame(baseline_rows)
baseline_df


### 🔍 结果透视：Baseline 到底检索到了什么？

在评估 RAG 效果时，我们不能只看 LLM 的最终回答（因为 LLM 可能会“脑补”知识）。**更科学的方法是直接观察检索出的上下文**，看它是否包含了回答问题所需的必要信息。

我们以第 0 个问题为例，看看 Baseline 检索到的 4 个片段：

In [ ]:
def inspect_retrieval(retriever, question):
    docs = retriever.invoke(question)
    print(f"❓ 问题: {question}\n")
    print(f"📦 检索到 {len(docs)} 个片段:\n")
    for i, doc in enumerate(docs):
        print(f"--- 片段 {i+1} ---\n")
        print(doc.page_content)
    print("\n" + "="*50 + "\n")

test_q = list(qna_dict.keys())[0]
inspect_retrieval(baseline_retriever, test_q)

### Baseline 失败分析

Baseline 使用较小字符块（96），往往 **能命中相关片段**，但定义与例证、前后句常被切断，送给 LLM 的上下文不完整。下面三种方法在 **同一测试集** 上补全上下文。


## Sentence Window（句子窗口检索）

> 对「无句号且过长」的片段再切段，减轻 PDF 目录被当作一整句、窗口拖入海量无关内容的情况。

回到 baseline 的失败案例：检索器命中了关键句，但紧接着的补充说明落在了下一个 chunk 里。问题不在检索，而在于**命中句的前后支撑句被丢掉了**。

### 核心思想

Sentence Window 的思路非常直觉——既然丢的是前后几句，那就在检索命中后把左右邻居补回来：

1. **索引时**：把文档按句子切分，每个句子单独嵌入，同时在元数据中记录该句子的前后邻居
2. **检索时**：先用句子级 embedding 做检索，命中后不直接把这一句送给 LLM，而是查邻居映射，把左右各 N 句拼上，再送给 LLM

### 与 Baseline 的区别

| 阶段 | Baseline | Sentence Window |
|---|---|---|
| 索引粒度 | 固定字符块 | 句子级 |
| 检索对象 | 整个 chunk | 单个句子 |
| 返回内容 | 命中的 chunk | 命中句子 + 前后窗口 |

窗口固定；证据若分散在不同段落，可配合 Small-to-Big。


In [ ]:
base_docs = PyMuPDFLoader(PDF_PATH).load()
for doc in base_docs:
    doc.page_content = clean_text(doc.page_content)
full_text = "\n".join(d.page_content for d in base_docs)

sentences = split_sentences_for_window(full_text)
print(f"总句子数: {len(sentences)}")

sentence_map = {i: s for i, s in enumerate(sentences)}
WINDOW_SIZE = 3
neighbor_map = {
    i: list(range(max(0, i - WINDOW_SIZE), min(len(sentences), i + WINDOW_SIZE + 1)))
    for i in sentence_map
}

persist_dir_sw = "./chroma_db/sentence_window_v2"
if os.path.exists(persist_dir_sw) and os.listdir(persist_dir_sw):
    sentence_vs = build_chroma_from_texts(None, embeddings, persist_directory=persist_dir_sw)
else:
    sentence_vs = build_chroma_from_texts(list(sentence_map.values()), embeddings, persist_directory=persist_dir_sw)
sentence_retriever = sentence_vs.as_retriever(search_kwargs={"k": 3})

def sentence_window_answer(question: str) -> str:
    hits = sentence_retriever.invoke(question)
    hit_texts = [h.page_content for h in hits]
    hit_ids = [idx for idx, txt in sentence_map.items() if txt in hit_texts]
    window_ids = sorted({nid for hid in hit_ids for nid in neighbor_map.get(hid, [hid])})
    window_context = "\n".join(sentence_map[i] for i in window_ids)
    prompt = f"仅根据上下文回答问题，如果上下文没有包含完整答案，请仅回答上下文中的内容，不要补充你自己的知识。请简洁作答。\n问题：{question}\n上下文：\n{window_context}"
    return llm_call(prompt)

rows = []
for question, expected in qna_dict.items():
    answer = sentence_window_answer(question)
    rows.append({
        "train_idx": question_to_train_idx[question],
        "question": question,
        "llm_answer": answer,
        "expected_answer": expected,
        "rag_eval_results": simple_eval(answer, expected, question),
    })

sentence_window_df = pd.DataFrame(rows)
sentence_window_df


### 🔍 结果透视：Sentence Window 增强了什么？

在这个方法中，检索出的实际上是**单个句子**。但我们并不直接用它，而是根据它的 `index` 向前和向后各取了 3 句。来看看这种变化是否真正补全了上下文：

In [ ]:
def inspect_sentence_window(question):
    hits = sentence_retriever.invoke(question)
    hit_texts = [h.page_content for h in hits]
    hit_ids = [idx for idx, txt in sentence_map.items() if txt in hit_texts]
    
    print(f"❓ 问题: {question}\n")
    for i, hid in enumerate(hit_ids):
        print(f"--- 命中句 {i+1} ---\n")
        print(f"[原始命中]: {sentence_map[hid]}")
        
        # 窗口内容
        window_ids = sorted({nid for nid in neighbor_map.get(hid, [hid])})
        window_text = "\n".join(sentence_map[idx] for idx in window_ids)
        print(f"[增强窗口]:\n{window_text}\n")
    print("="*50)

test_q = list(qna_dict.keys())[0]
inspect_sentence_window(test_q)

### Sentence Window 结果分析

**典型受益题（相对 Baseline）**：train 下标 **5**（交叉验证）。

**局限**：窗口固定；证据分散在不相邻段落时需段落级方案。


## Small-to-Big（父文档检索）

Sentence Window 能补回前后几句，但如果证据散落在段落的不同位置呢？比如段落开头有概念定义，中间有具体例子，结尾有对比总结——这些信息不在某一句的邻域里，而是分散在整个段落中。

### 核心思想

这就引出了一个经典的切块两难：**小块召回准但信息碎，大块信息全但相似度容易偏移**。Small-to-Big 用一个简单的分层策略同时解决这两个问题：

1. **索引时**：创建两级分块
   - **子块**：小块用于精确检索，嵌入向量存储在向量库
   - **父块**：大块用于提供完整上下文，存储在文档库
   - 记录每个子块属于哪个父块（`child_id → parent_id`）

2. **检索时**：
   - 用子块做精确召回定位
   - 不直接返回子块内容，而是通过映射找到它所属的父块
   - 把整个父块送给 LLM

### 与 Sentence Window 的区别

| 维度 | Sentence Window | Small-to-Big |
|---|---|---|
| 检索粒度 | 句子 | 子块（可自定义大小） |
| 上下文来源 | 固定窗口（前后 N 句） | 父块（语义完整的段落） |
| 灵活性 | 窗口大小固定 | 父块大小可调 |
| 适用场景 | 连续叙述文本 | 有清晰段落结构的文档 |

这样检索端享受小块的精度优势，生成端享受大块的完整性优势。适合有清晰段落结构的文档（教材、技术文档、论文）。局限在于：如果文档结构极不规则（比如对话记录、日志），父子映射本身就不可靠。

In [ ]:
from langchain_core.documents import Document

raw_docs = PyMuPDFLoader(PDF_PATH).load()
documents = [Document(page_content=clean_text(d.page_content), metadata=d.metadata) for d in raw_docs]

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=480, chunk_overlap=60,
    separators=["\n\n", "\n", "。", "；", "：", " ", ""],
)
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100, chunk_overlap=20,
    separators=["\n\n", "\n", "。", "；", "：", " ", ""],
)

parent_docs = parent_splitter.split_documents(documents)
parent_texts = [d.page_content for d in parent_docs]
print(f"父块数量: {len(parent_texts)}")

child_texts = []
child_to_parent = {}
for p_idx, p_doc in enumerate(parent_docs):
    for c in child_splitter.split_text(p_doc.page_content):
        c = c.strip()
        if not c:
            continue
        c_idx = len(child_texts)
        child_texts.append(c)
        child_to_parent[c_idx] = p_idx

print(f"子块数量: {len(child_texts)}")
print(f"平均每个父块的子块数: {len(child_texts) / max(1, len(parent_texts)):.1f}")

persist_dir_child = "./chroma_db/small_to_big_ragdoc"
if os.path.exists(persist_dir_child) and os.listdir(persist_dir_child):
    child_vs = build_chroma_from_texts(None, embeddings, persist_directory=persist_dir_child)
else:
    child_vs = build_chroma_from_texts(child_texts, embeddings, persist_directory=persist_dir_child)
child_retriever = child_vs.as_retriever(search_kwargs={"k": 4})

def small_to_big_answer(question: str) -> str:
    hits = child_retriever.invoke(question)
    hit_set = {h.page_content for h in hits}
    hit_ids = [idx for idx, txt in enumerate(child_texts) if txt in hit_set]
    parent_ids = sorted({child_to_parent[i] for i in hit_ids})
    context = "\n\n".join(parent_texts[i] for i in parent_ids[:3])
    prompt = f"仅根据上下文回答问题，如果上下文没有包含完整答案，请仅回答上下文中的内容，不要补充你自己的知识。请简洁作答。\n问题：{question}\n上下文：\n{context}"
    return llm_call(prompt)

rows = []
for question, expected in qna_dict.items():
    answer = small_to_big_answer(question)
    rows.append({
        "train_idx": question_to_train_idx[question],
        "question": question,
        "llm_answer": answer,
        "expected_answer": expected,
        "rag_eval_results": simple_eval(answer, expected, question),
    })

small_to_big_df = pd.DataFrame(rows)
small_to_big_df


### 🔍 结果透视：Small-to-Big 如何找回父文档？

Small-to-Big 的关键在于**检索子块，返回父块**。我们看看具体的子块命中情况以及对应的父块是否更完整：

In [ ]:
def inspect_small_to_big(question):
    hits = child_retriever.invoke(question)
    hit_texts = [h.page_content for h in hits]
    hit_ids = [idx for idx, txt in enumerate(child_texts) if txt in hit_texts]
    
    print(f"❓ 问题: {question}\n")
    for i, cid in enumerate(hit_ids):
        pid = child_to_parent[cid]
        print(f"--- 子块命中 {i+1} ---\n")
        print(f"[子块内容]: {child_texts[cid]}")
        print(f"[对应父块]:\n{parent_texts[pid]}\n")
    print("="*50)

inspect_small_to_big(test_q)

### Small-to-Big 结果分析

父/子块由 `RecursiveCharacterTextSplitter` 在按页清洗后的 `Document` 上切分，比整本书定长滑窗更接近小节边界。

**典型受益题**：train 下标 **0**（「算法 / 模型」与关系）。

**局限**：切分仍依赖 `chunk_size` 与分隔符；结构混乱时父块未必是语义段落。


## AutoMerging（自动合并检索）

Small-to-Big 里有一个隐含假设：每个 child 命中后，直接回填它所属的 parent 就行。但实际场景中经常出现这样的情况——**同一个 parent 下有好几个 child 都被命中了**。这时候如果按 Small-to-Big 的逻辑，同一个 parent 会被重复回填，既浪费 token 又可能引入冗余。

### 核心思想

AutoMerging 的思路是：与其被动回填，不如主动判断——**如果一个 parent 下被命中的 child 比例超过了阈值，就直接合并整个 parent 块作为上下文**。

具体流程：
1. **索引时**：和 Small-to-Big 一样，创建层级文档结构
2. **检索时**：
   - 先用叶子块（最小粒度块）做检索
   - 统计每个 parent 下有多少叶子被命中，算一个命中密度（命中数 / 总叶子数）
   - 密度超过阈值（比如 50%）的 parent，整体提升为最终上下文
   - 没超过的，退回到用单个叶子块

### 与 Small-to-Big 的区别

| 维度 | Small-to-Big | AutoMerging |
|---|---|---|
| 返回策略 | 每个命中子块都返回其父块 | 根据命中密度决定是否合并父块 |
| 重复问题 | 可能重复返回同一父块 | 自动去重合并 |
| 噪声控制 | 无 | 通过阈值过滤低相关性父块 |
| 复杂度 | 低 | 中（需要计算命中密度） |

这个阈值是 AutoMerging 的核心调控参数：设得太低，几乎所有 parent 都会被合并，噪声增多；设得太高，行为退化成 Small-to-Big。适合文档有明显层级结构、且证据经常分散在同一 parent 内多个位置的场景。

In [ ]:
leaf_per_parent = {}
for c_idx, p_idx in child_to_parent.items():
    leaf_per_parent.setdefault(p_idx, []).append(c_idx)

print(f"父块数量: {len(leaf_per_parent)}")
print(f"平均每个父块的子块数: {sum(len(v) for v in leaf_per_parent.values()) / len(leaf_per_parent):.1f}")

MERGE_THRESHOLD = 0.45

def auto_merge_answer(question: str, merge_threshold: float = MERGE_THRESHOLD) -> str:
    hits = child_retriever.invoke(question)
    hit_set = {h.page_content for h in hits}
    hit_ids = set(idx for idx, txt in enumerate(child_texts) if txt in hit_set)

    merged_parents = []
    sparse_parts = []
    seen_parent = set()

    for p_idx, leaves in leaf_per_parent.items():
        hit_count = sum(1 for lid in leaves if lid in hit_ids)
        if hit_count == 0:
            continue
        total = len(leaves)
        ratio = hit_count / max(1, total)
        if ratio >= merge_threshold:
            if p_idx not in seen_parent:
                merged_parents.append(parent_texts[p_idx])
                seen_parent.add(p_idx)
        else:
            for lid in leaves:
                if lid in hit_ids:
                    sparse_parts.append(child_texts[lid])

    parts = merged_parents + sparse_parts[:4]
    context = "\n\n".join(parts[:8])
    print(f"  -> 合并父块数: {len(merged_parents)}，稀疏子块片段: {min(len(sparse_parts), 4)}")

    prompt = f"仅根据上下文回答问题，如果上下文没有包含完整答案，请仅回答上下文中的内容，不要补充你自己的知识。请简洁作答。\n问题：{question}\n上下文：\n{context}"
    return llm_call(prompt)

rows = []
for question, expected in qna_dict.items():
    answer = auto_merge_answer(question, merge_threshold=MERGE_THRESHOLD)
    rows.append({
        "train_idx": question_to_train_idx[question],
        "question": question,
        "llm_answer": answer,
        "expected_answer": expected,
        "rag_eval_results": simple_eval(answer, expected, question),
    })

auto_merging_df = pd.DataFrame(rows)
auto_merging_df


### 🔍 结果透视：AutoMerging 如何实现自动合并？

AutoMerging 不只是简单的回填，它会计算**命中密度**。我们看看在同一个问题下，哪些父块被触发了合并逻辑：

In [ ]:
def inspect_auto_merging(question, threshold=0.45):
    hits = child_retriever.invoke(question)
    hit_texts = [h.page_content for h in hits]
    hit_ids = [idx for idx, txt in enumerate(child_texts) if txt in hit_texts]
    
    # 统计父块的命中情况
    parent_hit_counts = {}
    for cid in hit_ids:
        pid = child_to_parent[cid]
        parent_hit_counts[pid] = parent_hit_counts.get(pid, 0) + 1
    
    print(f"❓ 问题: {question}\n")
    for pid, count in parent_hit_counts.items():
        total_children = list(child_to_parent.values()).count(pid)
        ratio = count / total_children
        is_merged = ratio >= threshold
        
        print(f"--- 父块 {pid} (命中 {count}/{total_children}, 比例 {ratio:.1%}) ---\n")
        if is_merged:
            print(f"✅ [已合并]: 使用完整父块内容\n{parent_texts[pid][:200]}...")
        else:
            print(f"❌ [未合并]: 仅使用命中的子块片段")
    print("="*50)

inspect_auto_merging(test_q)



### AutoMerging 结果分析

仅当某父块下 **命中子块比例 ≥ MERGE_THRESHOLD** 时抬升整父块，否则只用被命中的子块片段。

**典型受益题**：train 下标 **26**（式(4.8) 与 λ）。

**局限**：依赖层级切分；阈值需调参。


## 同题对比总览（共用测试集）

将 Baseline 与各增强在同一题上的裁判结果并列，便于核对 **WIN_TARGETS** 中的典型胜场（增强严格优于 Baseline：Baseline ❌ 且 该方法 ✅）。裁判偏严时若未满足，以 inspect 与上下文完整性为准。


In [ ]:
def build_compare_table(baseline_df, sw_df, stb_df, am_df):
    key = "train_idx"
    b = baseline_df[[key, "question", "rag_eval_results"]].rename(columns={"rag_eval_results": "baseline"})
    m = b.merge(sw_df[[key, "rag_eval_results"]].rename(columns={"rag_eval_results": "sentence_window"}), on=key)
    m = m.merge(stb_df[[key, "rag_eval_results"]].rename(columns={"rag_eval_results": "small_to_big"}), on=key)
    m = m.merge(am_df[[key, "rag_eval_results"]].rename(columns={"rag_eval_results": "auto_merging"}), on=key)
    return m.sort_values(key).reset_index(drop=True)

compare_df = build_compare_table(baseline_df, sentence_window_df, small_to_big_df, auto_merging_df)
compare_df

print("\n--- 典型胜场核对（WIN_TARGETS）---")
for method, col in [("Sentence Window", "sentence_window"), ("Small-to-Big", "small_to_big"), ("AutoMerging", "auto_merging")]:
    tidx = WIN_TARGETS[method]
    row = compare_df.loc[compare_df["train_idx"] == tidx]
    if row.empty:
        print(f"  [{method}] train 下标 {tidx} 不在本批子集中")
        continue
    b = row["baseline"].iloc[0]
    x = row[col].iloc[0]
    ok = (b == "❌" and x == "✅")
    tag = "✅ 符合「胜 Baseline」" if ok else "ℹ️ 未满足严格判据（可对照上文 inspect）"
    print(f"  [{method}] train[{tidx}] baseline={b} {col}={x} -> {tag}")


## 实验结果汇总

**阅读顺序**：先看上表 **同题对比**，再看各 `*_df`。LLM 裁判偏严，不必以 ✅ 数量为唯一结论。

| train 下标 | 题意摘要 | 典型展示方法 |
|---|---|---|
| 0 | 「算法」与「模型」及关系 | Small-to-Big |
| 5 | 交叉验证 vs 单次留出 | Sentence Window |
| 14 | argmin 与 min | 短答对照 |
| 26 | 式(4.8) λ | AutoMerging |
| 27 | 图4-2 四次划分 | 决策树叙述 |

### 定性结论

- **Baseline**：小块，易碎。
- **Sentence Window**：轻量；注意目录/长段（本节已切段缓解）。
- **Small-to-Big**：小块定位 + 父块生成。
- **AutoMerging**：阈值在整父块与叶子片段间折中。


## 方法特性对比

| 方法 | 核心思想 | 典型修复问题 | 新增复杂度 | 最适合文档形态 |
|---|---|---|---|---|
| baseline | 无（作为对照） | 无 | 低 | 任意 |
| Sentence Window | 命中句子后扩展前后窗口 | 命中句缺邻域证据 | 低 | 连续叙述文本 |
| Small-to-Big | 子块检索，父块生成 | 小块准但不完整 | 中 | 段落层级清晰 |
| AutoMerging | 根据命中密度合并父块 | 多子块分散命中同一父块 | 中-高 | 树状层级明显 |
| Late Chunking（理论） | 索引时让 chunk 看到完整文档 | embedding 缺少文档全局信息 | 低（但模型受限） | 需长上下文 embedding 模型 |

## 前沿方法：Late Chunking（理论介绍）

### 传统流程的问题
传统 RAG 的流程是先切分，再分别嵌入——每个 chunk 独立通过 embedding 模型，丢失了跨 chunk 的语义关联。

### Late Chunking 思路
Late Chunking 颠倒了这个顺序：
1. 先将**整个文档**送入长上下文 embedding 模型（如 jina-embeddings-v2），获得每个 token 的上下文化表示
2. 再按预设边界切分 token embeddings，对每个 chunk 的 token embeddings 做池化得到 chunk embedding

这样每个 chunk 的向量都见过完整文档上下文，天然缓解了上下文割裂问题。

### 与本章方法的对比

| 维度 | Sentence Window / Small-to-Big / AutoMerging | Late Chunking |
|---|---|---|
| 增强时机 | 检索时（命中后恢复邻域） | 索引时（embedding 阶段） |
| 额外存储 | 需要维护邻居映射 / 父子关系 | 不需要 |
| 模型依赖 | 无特殊要求 | 需要长上下文 embedding 模型 |
| 实现复杂度 | 中 | 低（但模型选择受限） |

### 局限
- 依赖支持长上下文的 embedding 模型（如 jina-embeddings-v2、nomic-embed），智谱 embedding-3 等通用 embedding 模型不直接支持此模式
- 文档超过模型上下文窗口时需要分段处理
- 目前 LangChain 生态无开箱即用的 Late Chunking 组件

### 本节为什么不做代码示例
Late Chunking 需要特殊的 embedding 模型和自定义 tokenizer 操作，与本节统一使用本地 bge-small-zh-v1.5 的可复现实验设定不兼容。此处仅作概念介绍，帮助读者建立还有一类索引时上下文增强的认知。

## 如何选择

- 如果问题经常只差前后两句：先用 Sentence Window。
- 如果文档天然有章节层级：优先 Small-to-Big。
- 如果证据常分散在同一父块多个子块：优先 AutoMerging。
- 若问题本质是多步推理或多轮交互，应转到流程增强或系统增强。
- 如果问题出在 chunk 本身缺少文档语境（脱离上下文后语义不完整），这属于**索引阶段**优化，应回到第 3 章的 CCH / Contextual Retrieval。本章的上下文增强解决的是检索后恢复邻域，而非索引时缺少语境。

## 下一步

如果你发现问题的根源不是上下文不全，而是一次检索流程本身不够——比如需要多步推理、需要先评估检索质量再决定下一步——请继续学习 `2. 流程增强.ipynb`。

### 学习检查点

- 你能区分检索相关但上下文不足与流程不足吗？
- 你能解释 Sentence Window 与 Small-to-Big 的关键差异吗？
- 你知道 AutoMerging 的阈值会如何影响召回上下文长度吗？